In [1]:
#Import the necessary libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn import datasets
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    precision_score,
    recall_score
)
#Reads the dataset from Excel
df = pd.read_excel("marketing_campaign.xlsx")

#Checks the first 5 rows to make sure that the data were loaded successfully
df.head()


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


This is the section where it checks for any missing data, if there are, and the data is cleaned up

In [2]:
#Checks if there are any missing data in it or duplicate data
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)
print("\n")
print(df.isnull().sum())
print("\n")
print((df == "?").sum())


Number of duplicate rows: 0


ID                      0
Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
dtype: int64


ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Kidhome                0
Teenhome               0
Dt_Customer            0
R

In [3]:

#Replaces all of recognizable missing data in the income column with median of it
median_value = df['Income'].median()
df['Income'] = df['Income'].fillna(median_value)

#Get rid of any data that could negatively impact the models or doesn't offer much to them
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue", "Dt_Customer"],errors="ignore")


#Gets all of the categorical columns and uses OneHot Encoding on them through the get_dummies functions
categorical_col = df.select_dtypes("object").columns.tolist()
if categorical_col:
    df = pd.get_dummies(df, columns = categorical_col,drop_first=True,dtype = int)



print(df.isnull().sum())
print("\n")
print((df == "?").sum())

Year_Birth                 0
Income                     0
Kidhome                    0
Teenhome                   0
Recency                    0
MntWines                   0
MntFruits                  0
MntMeatProducts            0
MntFishProducts            0
MntSweetProducts           0
MntGoldProds               0
NumDealsPurchases          0
NumWebPurchases            0
NumCatalogPurchases        0
NumStorePurchases          0
NumWebVisitsMonth          0
AcceptedCmp3               0
AcceptedCmp4               0
AcceptedCmp5               0
AcceptedCmp1               0
AcceptedCmp2               0
Complain                   0
Response                   0
Education_Basic            0
Education_Graduation       0
Education_Master           0
Education_PhD              0
Marital_Status_Alone       0
Marital_Status_Divorced    0
Marital_Status_Married     0
Marital_Status_Single      0
Marital_Status_Together    0
Marital_Status_Widow       0
Marital_Status_YOLO        0
dtype: int64



In [4]:

#This is the target variable that we are trying to predict
y = df['Response']

#The predictors used to predict the target variable
x = df.drop(columns = ['Response'])

#Splits the dataset up with 70% as training data and the rest as test datasets
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state =42)

# Find columns containing only 0 and 1 so that they aren't standardized
binary_columns = [
    column for column in x_train.columns
    if x_train[column].dropna().isin([0, 1]).all()
]

# Find columns that aren't containing only 0 and 1 where they would be standardized
nonbinary_columns = [
    column for column in x_train.columns
    if column not in binary_columns
]

#Prints the binary and nonbinary columns out to see if got them correct
print("Binary columns:", binary_columns)
print("\nColumns being scaled:", nonbinary_columns)

# Creates independent copies so that the original training and test
# datasets remain unchanged.
x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()

# Creates the StandardScaler object.
scaler = StandardScaler()

# Calculates each nonbinary training variable's mean and standard
# deviation and transforms the training values to a common scale.
x_train_scaled[nonbinary_columns] = scaler.fit_transform(
    x_train[nonbinary_columns]
)

#Standardizes the test data using the means and standard deviations as well
x_test_scaled[nonbinary_columns] = scaler.transform(
    x_test[nonbinary_columns]
)

Binary columns: ['AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Education_Basic', 'Education_Graduation', 'Education_Master', 'Education_PhD', 'Marital_Status_Alone', 'Marital_Status_Divorced', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Widow', 'Marital_Status_YOLO']

Columns being scaled: ['Year_Birth', 'Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']


Part 2: Now let's look at Logistic Regression's perforamnce

In [5]:
# Creates and trains a logistic regression model using the training data.
#the random state makes the model's result reproducible,with the solver as the liblinear

LogReg = LogisticRegression(solver = 'liblinear', random_state = 0).fit(x_train_scaled,y_train.values.ravel())

# Calculates the model's accuracy on the training data.
print(f"The training Logistic Regression model's accuracy: {LogReg.score(x_train_scaled,y_train):.3f}")


The training Logistic Regression model's accuracy: 0.893


In [6]:
#Creates and trains the logistic regression using statsmodel for the logistic regression model
#so that the predictors' coefficients and p-values can be examined
est = sm.Logit(y_train, x_train_scaled).fit()

#Creates a table that contains the predictors' coefficients and p-values.
#The coefficient shows the direction of the variable's influence
#while the p-value indicates whether the variable is statistically significant
results = pd.DataFrame({
    "Coefficient": est.params,
    "P-value": est.pvalues
})

#Sorts the variables from the smallest to largest p-value that are less than 0.05
significant_results = (
    results[results["P-value"] < 0.05]
    .sort_values(by="P-value")
)


#Displays significant predictors with p-values less than 0.05 with their coefficient
print("The significant variables of the Logistic Regression:")
display(significant_results)

Optimization terminated successfully.
         Current function value: 0.262616
         Iterations 8
The significant variables of the Logistic Regression:


,Coefficient,P-value
Marital_Status_Together,-4.263578,4.089588e-22
Marital_Status_Married,-4.003774,6.957573e-22
Recency,-0.845423,1.201475e-16
Marital_Status_Single,-2.969568,1.233128e-12
AcceptedCmp3,1.886400,1.275485e-12
Marital_Status_Divorced,-2.970111,1.456109e-11
AcceptedCmp5,1.992760,4.583170e-09
Marital_Status_Widow,-2.812643,5.384573e-07
AcceptedCmp1,1.352783,3.952561e-05
AcceptedCmp4,1.412685,4.079211e-05


In [7]:
#This cell now tests the model with the test data sets

#Gets the probability of a customer accepting the offer in the last campaign
y_probability = LogReg.predict_proba(x_test_scaled)[:, 1]

# Set the classification threshold
threshold = [0.40, 0.80]


for i in range(len(threshold)):

    print(f"\nResults for threshold = {threshold[i]}")
    
    # An observation is classified as 1 if
    #its predicted probability is at least greater than or equal to the threshold
    y_pred = (y_probability >= threshold[i]).astype(int)

    #Get the confusion matrix of the true negative and positive as well as the
    #false negative and positive
    print("Confusion Matrix for Logistic Regression:")
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
    print("True Negative: ",tn) 
    print("False Positive: ",fp)
    print("False Negative: ",fn)
    print("True Positive:", tp)

    #Displays the precision, recall, F1-score, and support for each class.
    #Also displays the total accuracy of the model
    print("\nClassification Report for Logistic Regression:")
    print(classification_report(y_test, y_pred))

    #This also calculates and displays the accuracy of the prediction from the model by
    # comparing the test data with predictions
    print(f"The accuracy of the Logistic Regression model using test data: {accuracy_score(y_test,y_pred):.3f}")



Results for threshold = 0.4
Confusion Matrix for Logistic Regression:
True Negative:  547
False Positive:  30
False Negative:  54
True Positive: 41

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       577
           1       0.58      0.43      0.49        95

    accuracy                           0.88       672
   macro avg       0.74      0.69      0.71       672
weighted avg       0.86      0.88      0.87       672

The accuracy of the Logistic Regression model using test data: 0.875

Results for threshold = 0.8
Confusion Matrix for Logistic Regression:
True Negative:  574
False Positive:  3
False Negative:  76
True Positive: 19

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.88      0.99      0.94       577
           1       0.86      0.20      0.32        95

    accuracy                           0.88    

Part3: Now let's look at Support Vector Machine Regression performance.

In [8]:
#Train the SVM model with the training data
svm_model = svm.SVC(kernel = 'linear',probability=True, random_state = 42)
svm_train = svm_model.fit(x_train_scaled,y_train.values.ravel())
print(f"Training Accuracy for SVM: {svm_train.score(x_train_scaled,y_train):.3f}")

Training Accuracy for SVM: 0.895


In [9]:
#Displays the coefficients of the variables in the SVM model

# Obtains the coefficients learned by the linear SVM.
# A positive coefficient pushes a prediction toward Response = 1,
# while a negative coefficient pushes it toward Response = 0.
svm_coefficient_results = pd.DataFrame({
    "Variable": x_train_scaled.columns,
    "Coefficient": svm_train.coef_[0]
})

# Uses the absolute coefficient to measure the strength of
# each variable regardless of whether its direction is positive or negative.
svm_coefficient_results["Absolute Coefficient"] = (
    svm_coefficient_results["Coefficient"].abs()
)

# Places variables with the largest coefficient magnitudes first.
svm_coefficient_results = (
    svm_coefficient_results
    .sort_values(
        by="Absolute Coefficient",
        ascending=False
    )
)

# Displays the ten variables with the largest coefficients.
display(
    svm_coefficient_results.head(10).style.format({
        "Coefficient": "{:.4f}",
        "Absolute Coefficient": "{:.4f}"
    })
)

,Variable,Coefficient,Absolute Coefficient
18,AcceptedCmp5,1.2724,1.2724
16,AcceptedCmp3,1.1873,1.1873
19,AcceptedCmp1,0.8822,0.8822
17,AcceptedCmp4,0.8604,0.8604
20,AcceptedCmp2,0.7882,0.7882
30,Marital_Status_Together,-0.6508,0.6508
25,Education_PhD,0.5927,0.5927
4,Recency,-0.4336,0.4336
28,Marital_Status_Married,-0.3935,0.3935
24,Education_Master,0.3776,0.3776


In [10]:
#This cell checks the importance of each variable

# Measures how much the test F1-score decreases when each
# predictor is randomly shuffled.
#
# F1-score is used because Response = 1 is much less common
# than Response = 0.
permutation_result = permutation_importance(
    svm_train,
    x_test_scaled,
    y_test.values.ravel(),
    n_repeats=30,
    random_state=42,
    scoring="f1"
)

# Creates a table containing the average importance and
# its variation across the repeated permutations.
svm_permutation_results = pd.DataFrame({
    "Variable": x_test_scaled.columns,
    "Importance": permutation_result.importances_mean
})

# Places the most influential variables first.
svm_permutation_results = (
    svm_permutation_results
    .sort_values(
        by="Importance",
        ascending=False
    )
)

# Displays the ten most important variables.
display(
    svm_permutation_results.head(10).style.format({
        "Importance": "{:.4f}"
    })
)

,Variable,Importance
16,AcceptedCmp3,0.1027
18,AcceptedCmp5,0.0605
4,Recency,0.0571
17,AcceptedCmp4,0.0319
30,Marital_Status_Together,0.0283
19,AcceptedCmp1,0.0277
3,Teenhome,0.0255
14,NumStorePurchases,0.0249
15,NumWebVisitsMonth,0.0239
24,Education_Master,0.0226


In [11]:
#This cell now tests the SVM model with the test datasets

#Gets the probability of a customer accepting the offer in the last campaign for SVM
svm_probability = svm_train.predict_proba(x_test_scaled)[:, 1]

# Set the classification threshold
threshold = [0.40, 0.80]

for i in range(len(threshold)):

    print(f"\nResults for threshold = {threshold[i]}")
    
    # An observation is classified as 1 if
    #its predicted probability is at least greater than or equal to the threshold
    y_pred2 = (svm_probability >= threshold[i]).astype(int)

    #Get the confusion matrix of the true negative and positive as well as the
    #false negative and positive
    print("Confusion Matrix of SVM:")
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred2).ravel().tolist()
    print("True Negative: ",tn) 
    print("False Positive: ",fp)
    print("False Negative: ",fn)
    print("True Positive:", tp)

    #Displays the precision, recall, F1-score, and support for each class.
    #Also displays the total accuracy of the model
    print("\nClassification Report for SVM:")
    print(classification_report(y_test, y_pred2))

    #This also calculates and displays the accuracy of the prediction from the model by
    # comparing the test data with predictions
    print(f"The accuracy of the SVM Regression model using test data: {accuracy_score(y_test,y_pred2):.3f}")




Results for threshold = 0.4
Confusion Matrix of SVM:
True Negative:  553
False Positive:  24
False Negative:  57
True Positive: 38

Classification Report for SVM:
              precision    recall  f1-score   support

           0       0.91      0.96      0.93       577
           1       0.61      0.40      0.48        95

    accuracy                           0.88       672
   macro avg       0.76      0.68      0.71       672
weighted avg       0.87      0.88      0.87       672

The accuracy of the SVM Regression model using test data: 0.879

Results for threshold = 0.8
Confusion Matrix of SVM:
True Negative:  577
False Positive:  0
False Negative:  84
True Positive: 11

Classification Report for SVM:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       577
           1       1.00      0.12      0.21        95

    accuracy                           0.88       672
   macro avg       0.94      0.56      0.57       672
weighted a

Now let's see and compare the two models based on their accuracy, precision and recall

In [12]:
# Classification thresholds to compare
selected_thresholds = [0.40, 0.80]

# Stores one results row for each model and threshold
comparison_rows = []

for current_threshold in selected_thresholds:

    # Creates logistic-regression predictions at the current threshold
    logistic_final_prediction = (
        y_probability >= current_threshold
    ).astype(int)

    # Creates SVM predictions at the current threshold
    svm_final_prediction = (
        svm_probability >= current_threshold
    ).astype(int)

    # Calculates metrics for logistic regression
    comparison_rows.append({
        "Model": "Logistic Regression",
        "Threshold": current_threshold,
        "Accuracy": accuracy_score(
            y_test,
            logistic_final_prediction
        ),
        "Precision": precision_score(
            y_test,
            logistic_final_prediction
        ),
        "Recall": recall_score(
            y_test,
            logistic_final_prediction
        )
    })

    # Calculates metrics for the linear SVM
    comparison_rows.append({
        "Model": "Linear SVM",
        "Threshold": current_threshold,
        "Accuracy": accuracy_score(
            y_test,
            svm_final_prediction
        ),
        "Precision": precision_score(
            y_test,
            svm_final_prediction
        ),
        "Recall": recall_score(
            y_test,
            svm_final_prediction
        )
    })

# Converts all model and threshold results into a DataFrame
model_comparison = pd.DataFrame(comparison_rows)

# Displays the comparison table
display(
    model_comparison.style.format({
        "Threshold": "{:.2f}",
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}"
    })
)

,Model,Threshold,Accuracy,Precision,Recall
0,Logistic Regression,0.40,0.875,0.577,0.432
1,Linear SVM,0.40,0.879,0.613,0.400
2,Logistic Regression,0.80,0.882,0.864,0.200
3,Linear SVM,0.80,0.875,1.000,0.116
